# Quantum Multi-Omic Classifier — Genotype-to-Phenotype Mapping via Data Re-Uploading

_Investigation `quantum-multi-omic-classifier` — coder reproduction notebook._

**Question.** Can a multi-qubit, cross-modality entangled quantum classifier outperform classical ML baselines in predicting binary phenotypes from synthetic multi-omic data?

An investigation into whether a purpose-built multi-qubit quantum classifier can improve phenotype prediction from multi-omic data by using entanglement to model biological cross-talk between molecular layers.

---

**How to use this notebook**

1. **Run ▸ Run All Cells** to reproduce every study end to end.
2. To explore, edit a study's **Parameters** cell, then re-run from its **Run** section — the figures regenerate with your values.
3. The **Setup** cell at the top is collapsed scaffolding (environment + render helpers); expand it only to inspect. Set `RERUN = False` there to render the committed `runs.db` without re-simulating.

The prose states the **question and parameters**; the figures each run produces are the results.


In [ ]:
"""Setup — environment & helpers (runs automatically; open only to inspect).

Generated by vivarium-workbench (notebook_export). This cell resolves the repo,
registers the workspace Core, and defines the render helpers; the science lives
in the sections below. Each study is re-run live with the workspace's own
process-bigraph protocol and its figures rendered from the resulting runs.db.
"""
import os
import sys
from pathlib import Path

# Resolve the repository root robustly so this notebook runs from a fresh clone
# at ANY path with no setup (no env var, no path editing). Priority:
#   1. $VIVARIUM_REPO, if it points at a real directory;
#   2. walk up from the notebook's working directory for the repo markers
#      (a directory holding both 'workspace/' and 'pyproject.toml') — Jupyter
#      starts in the notebook's dir, so a committed notebook finds its own root;
#   3. the absolute path it was generated for (back-compat for old layouts);
#   4. the current working directory (last resort).
def _find_repo_root(_start):
    for _cand in (_start, *_start.parents):
        if (_cand / "workspace").is_dir() and (_cand / "pyproject.toml").is_file():
            return _cand
    return None

REPO = None
_env = os.environ.get("VIVARIUM_REPO")
if _env and Path(_env).is_dir():
    REPO = Path(_env)
if REPO is None:
    REPO = _find_repo_root(Path.cwd().resolve())
if REPO is None and Path('/home/runner/work/viva-pennylane-data-reuploading/viva-pennylane-data-reuploading').is_dir():
    REPO = Path('/home/runner/work/viva-pennylane-data-reuploading/viva-pennylane-data-reuploading')
if REPO is None:
    REPO = Path.cwd()
sys.path.insert(0, str(REPO))
# Composite specs use repo-root-relative paths (datasets, caches), and the
# workspace's runner/renderer assume cwd == repo root — so run from there.
os.chdir(REPO)

# Re-simulate from scratch? Set False to render the committed runs.db (fast).
RERUN = True

# Keep the output clean: composite discovery and the pydantic models emit
# benign UserWarnings (optional packages it skips, a field-name shadow) and the
# simulators emit DeprecationWarnings — none are actionable from a reproduction
# notebook, so silence those two categories. Remove this to see them.
import warnings as _warnings
_warnings.filterwarnings("ignore", category=UserWarning)
_warnings.filterwarnings("ignore", category=DeprecationWarning)

# --- standard process-bigraph protocol: register the workspace's Core ---
from pbg_pennylane_data_reuploading.core import build_core
core = build_core()

# --- imported from the repo this notebook was generated for ---

from IPython.display import HTML, display

import contextlib as _contextlib, io as _io
@_contextlib.contextmanager
def quiet():
    """Silence the simulator's verbose per-step stdout so the notebook
    output stays readable (the figures below are the results)."""
    with _contextlib.redirect_stdout(_io.StringIO()):
        yield

import html as _htmlmod
def show_viz(_h, height=560):
    """Display a visualization's HTML in an isolated iframe.

    The figures embed their own scripts (e.g. Plotly); JupyterLab does not
    execute <script> tags from display(HTML(...)), so an iframe srcdoc is
    used instead — the browser runs the scripts inside the frame."""
    display(HTML(
        '<iframe srcdoc="{}" style="width:100%;height:{}px;border:0">'
        '</iframe>'.format(_htmlmod.escape(_h, quote=True), height)
    ))

import json as _json
def describe_spec(spec):
    """Print a composite spec's structure (parameters, processes, wiring)
    then the full editable dict. The spec is plain data — assign to any
    field (e.g. spec['state'][proc]['config'][...]) before building."""
    print("composite:", spec.get("name"))
    if spec.get("description"):
        print("description:", str(spec["description"]).strip())
    _params = spec.get("parameters") or {}
    if _params:
        print("\nparameters (filled into ${name} placeholders):")
        for _p, _pdef in _params.items():
            print(f"  {_p}: default={_pdef.get('default')!r}  type={_pdef.get('type')}")
    print("\nprocesses (node -> address):")
    for _node, _body in (spec.get("state") or {}).items():
        if not (isinstance(_body, dict) and _body.get("_type") == "process"):
            continue
        print(f"  {_node}  ->  {_body.get('address')}   interval={_body.get('interval')!r}")
        for _port in ("inputs", "outputs"):
            if _body.get(_port):
                print(f"      {_port} ports: {_body[_port]}")
    print("\nfull editable spec dict:")
    print(_json.dumps(spec, indent=2, default=str))

def materialize_composite(spec_id, overrides=None):
    """Build a composite into an editable payload {name, description,
    parameters, state, _overrides, _id}. Works for @composite_generator
    functions via the process-bigraph registry (the common case); ``state``
    is the built document (process nodes at top level)."""
    import importlib
    from process_bigraph.composite_generator import (
        _REGISTRY, apply_core_extensions, build_generator)
    try:
        from process_bigraph.composite_generator import discover_generators
        discover_generators()
    except Exception:
        pass
    entry = _REGISTRY.get(spec_id)
    if entry is None:
        # editable installs are invisible to packages_distributions, so
        # discover alone can miss them — prime the module the id names.
        try:
            importlib.import_module(spec_id.rsplit('.', 1)[0])
            from process_bigraph.composite_generator import discover_generators as _dg
            _dg()
        except Exception:
            pass
        entry = _REGISTRY.get(spec_id)
    if entry is None:
        raise KeyError('composite not registered: ' + repr(spec_id))
    params = dict(entry.parameters or {})
    ov = {k: (v.get('default') if isinstance(v, dict) else None)
          for k, v in params.items()}
    if overrides:
        ov.update({k: val for k, val in overrides.items() if k in params})
    _c = apply_core_extensions(entry, core)
    try:
        doc = build_generator(entry, ov or None, core=_c)
    except TypeError:
        doc = entry.func(_c, **(ov or {}))
    return {'name': entry.name, 'description': entry.description,
            'parameters': params, 'state': doc, '_overrides': ov, '_id': spec_id}

import base64 as _b64, pathlib as _pl
_STATIC_VIZ_SCHEMES = ('image:', 'file:', 'gif:', 'png:', 'svg:', 'jpg:', 'jpeg:')
def _render_viz_process(address, config):
    """Render a Visualization process/step (e.g. ``local:<Class>``) to
    HTML: wrap it as a one-step composite and run it once — the viz writes
    Plotly HTML to its ``html`` output port. Mirrors the workbench's own
    env-worker viz path. Self-running vizes need no run data."""
    from process_bigraph import Composite
    _doc = {
        'inputs_store': {}, 'output_store': '',
        'visualization': {
            '_type': 'step', 'address': address,
            'config': {k: v for k, v in (config or {}).items()
                       if k not in ('inputs_map', 'sources')},
            'inputs': {}, 'outputs': {'html': ['output_store']},
        },
    }
    _comp = Composite({'state': _doc}, core=core)
    _comp.run(1)
    _html = _comp.state.get('output_store')
    if isinstance(_html, dict):
        _html = _html.get('value') or _html.get('_value') or ''
    return _html if isinstance(_html, str) else ''
def _render_one(address, config, runs_db, study_yaml, overrides=None):
    """Generic figure renderer (no workspace render_study_viz.py).
    A static ``image:``/``file:`` (or bare-path) address embeds the file;
    any other address (``local:<VizProcess>``) is run as a Visualization
    and its HTML returned. ``overrides`` (the composite's edited parameter
    dict) is merged into the viz config first so a self-running figure
    reflects edited parameters; a viz that does not accept those keys
    falls back to the base config."""
    addr = str(address or '')
    _head = addr.split('/', 1)[0]
    _is_static = (not addr) or (':' not in _head) or addr.startswith(_STATIC_VIZ_SCHEMES)
    if not _is_static:
        if core is None:
            return f'<p style="color:#b91c1c">no core to render {address}</p>'
        _base = {k: v for k, v in (config or {}).items()}
        _tries = ([{**_base, **overrides}] if overrides else []) + [_base]
        _err = ''
        for _cfg in _tries:
            try:
                _h = _render_viz_process(addr, _cfg)
            except Exception as _e:
                _err = (f'<p style="color:#b91c1c">visualization {address} not renderable yet: '
                        f'{type(_e).__name__}: {_e}</p>')
                continue
            return _h or f'<p style="color:#6b7280">{address} produced no HTML</p>'
        return _err
    for _scheme in _STATIC_VIZ_SCHEMES:
        if addr.startswith(_scheme):
            addr = addr[len(_scheme):]; break
    _p = _pl.Path(addr)
    if not _p.is_absolute():
        _p = _pl.Path(study_yaml).resolve().parent / _p
    if not _p.is_file():
        return f'<p style="color:#b91c1c">figure not found: {address}</p>'
    _suffix = _p.suffix.lower()
    if _suffix == '.svg':
        return _p.read_text(encoding='utf-8', errors='replace')
    if _suffix in ('.png', '.jpg', '.jpeg', '.gif', '.webp'):
        _mime = 'jpeg' if _suffix in ('.jpg', '.jpeg') else _suffix[1:]
        _data = _b64.b64encode(_p.read_bytes()).decode('ascii')
        return f'<img src="data:image/{_mime};base64,{_data}" style="max-width:100%"/>'
    if _suffix in ('.html', '.htm'):
        return _p.read_text(encoding='utf-8', errors='replace')
    return f'<p style="color:#6b7280">unsupported figure type: {address}</p>'

## Study: Classical Baselines — Random Forest and Logistic Regression on Synthetic Multi-Omic Data (`genotype-phenotype-baseline`)

**Question.** What is the best AUC-ROC and F1 score achievable with classical ML methods on synthetic multi-omic phenotype prediction, to establish a baseline for quantum classifier comparison?

**Objective.** Train Random Forest and Logistic Regression on synthetic multi-omic data
and report accuracy, AUC-ROC, F1, precision, and recall.

**Hypothesis.** Random Forest will outperform Logistic Regression by capturing nonlinear cross-modality interactions, but both will serve as strong baselines.


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `rf_lr_baseline` | `pbg_pennylane_data_reuploading.composites.genotype_phenotype.genotype_to_phenotype_mapping` | 0 | num_omic_modalities=4, features_per_modality=6, epochs=6, batch_size=16, num_train=60, num_test=100, seed=42, noise_level=0.15, learning_rate=0.4 |


### The model & its parameters

Each composite is a process-bigraph *document* — named processes wired by `inputs`/`outputs` ports over shared stores. Below, each composite shows what it does, then a **parameter cell** with its tunable knobs (the current default and what each one means). Edit any value, then re-run the **Run** section to rebuild and re-simulate with your changes.


**The model — `pbg_pennylane_data_reuploading.composites.genotype_phenotype.genotype_to_phenotype_mapping`**

Classify multi-omic expression profiles (transcriptomics, proteomics, metabolomics, epigenomics) into viable vs non-viable cell replication using quantum data re-uploading. Each omic modality occupies one re-uploading layer.


In [ ]:
# === Parameters — edit any value, then re-run from the Run cell below ===
# Each entry is the composite's current default; the comment explains it.
ov_pbg_pennylane_data_reuploading_composites_genotype_phenotype_genotype_to_phenotype_mapping = {
    'num_omic_modalities'  : 4,   # Number of omic input modalities
    'features_per_modality': 8,   # Features per modality
    'learning_rate'        : 0.4,   # Adam learning rate
    'epochs'               : 8,   # Training epochs
    'batch_size'           : 16,   # Mini-batch size
}

### Run

_Set `STEPS`, then run. Each composite is built live from its (edited) spec above and run; a composite whose module processes aren't implemented yet reports the error instead of crashing. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: genotype-phenotype-baseline ===
STUDY = 'genotype-phenotype-baseline'
STUDY_DIR = REPO / 'workspace/studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

# This study has no *recorded* runs yet. RERUN builds its declared
# composite(s) live and runs them; a composite whose module processes
# are not implemented yet reports the error instead of crashing.
STEPS_rf_lr_baseline = 100
INTERVAL_rf_lr_baseline = 1.0

if RERUN:
    from process_bigraph import Composite
    try:
        with quiet():  # keep per-step stdout out of the notebook
            _ov = ov_pbg_pennylane_data_reuploading_composites_genotype_phenotype_genotype_to_phenotype_mapping if 'ov_pbg_pennylane_data_reuploading_composites_genotype_phenotype_genotype_to_phenotype_mapping' in dir() else None  # edits from the panel cell, if any
            spec_pbg_pennylane_data_reuploading_composites_genotype_phenotype_genotype_to_phenotype_mapping = materialize_composite('pbg_pennylane_data_reuploading.composites.genotype_phenotype.genotype_to_phenotype_mapping', _ov)
            _comp = Composite({'state': spec_pbg_pennylane_data_reuploading_composites_genotype_phenotype_genotype_to_phenotype_mapping['state']}, core=core)
            _comp.run(STEPS_rf_lr_baseline)
        print('ran', 'rf_lr_baseline', f'(pbg_pennylane_data_reuploading.composites.genotype_phenotype.genotype_to_phenotype_mapping) for', STEPS_rf_lr_baseline, 'steps')
    except Exception as _e:
        print(f"rf_lr_baseline not runnable yet: {type(_e).__name__}: {_e}")
else:
    print("RERUN=False — not re-simulating.")

## Study: Quantum Classifier Evaluation — GenotypeToPhenotypeProcess on Synthetic Multi-Omic Data (`quantum-classifier-evaluation`)

**Question.** Can the GenotypeToPhenotypeProcess match or exceed classical ML baselines in predicting binary phenotypes from synthetic multi-omic data?

**Objective.** Train the GenotypeToPhenotypeProcess with M=4 modalities, F=6 features per
modality on synthetic data, and compare accuracy, AUC-ROC, F1, precision,
and recall against RF and LR baselines.

**Hypothesis.** The quantum classifier will achieve competitive AUC-ROC (>= 0.80) on synthetic data with strong cross-modality interactions, benefiting from the CNOT entanglement chain.


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `quantum_4modality_6feat` | `pbg_pennylane_data_reuploading.composites.genotype_phenotype.genotype_to_phenotype_mapping` | 0 | num_omic_modalities=4, features_per_modality=6, epochs=6, batch_size=16, num_train=60, num_test=100, seed=42, noise_level=0.15, learning_rate=0.4 |


### The model & its parameters

Each composite is a process-bigraph *document* — named processes wired by `inputs`/`outputs` ports over shared stores. Below, each composite shows what it does, then a **parameter cell** with its tunable knobs (the current default and what each one means). Edit any value, then re-run the **Run** section to rebuild and re-simulate with your changes.


**The model — `pbg_pennylane_data_reuploading.composites.genotype_phenotype.genotype_to_phenotype_mapping`**

Classify multi-omic expression profiles (transcriptomics, proteomics, metabolomics, epigenomics) into viable vs non-viable cell replication using quantum data re-uploading. Each omic modality occupies one re-uploading layer.


In [ ]:
# === Parameters — edit any value, then re-run from the Run cell below ===
# Each entry is the composite's current default; the comment explains it.
ov_pbg_pennylane_data_reuploading_composites_genotype_phenotype_genotype_to_phenotype_mapping = {
    'num_omic_modalities'  : 4,   # Number of omic input modalities
    'features_per_modality': 8,   # Features per modality
    'learning_rate'        : 0.4,   # Adam learning rate
    'epochs'               : 8,   # Training epochs
    'batch_size'           : 16,   # Mini-batch size
}

### Run

_Set `STEPS`, then run. Each composite is built live from its (edited) spec above and run; a composite whose module processes aren't implemented yet reports the error instead of crashing. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: quantum-classifier-evaluation ===
STUDY = 'quantum-classifier-evaluation'
STUDY_DIR = REPO / 'workspace/studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

# This study has no *recorded* runs yet. RERUN builds its declared
# composite(s) live and runs them; a composite whose module processes
# are not implemented yet reports the error instead of crashing.
STEPS_quantum_4modality_6feat = 100
INTERVAL_quantum_4modality_6feat = 1.0

if RERUN:
    from process_bigraph import Composite
    try:
        with quiet():  # keep per-step stdout out of the notebook
            _ov = ov_pbg_pennylane_data_reuploading_composites_genotype_phenotype_genotype_to_phenotype_mapping if 'ov_pbg_pennylane_data_reuploading_composites_genotype_phenotype_genotype_to_phenotype_mapping' in dir() else None  # edits from the panel cell, if any
            spec_pbg_pennylane_data_reuploading_composites_genotype_phenotype_genotype_to_phenotype_mapping = materialize_composite('pbg_pennylane_data_reuploading.composites.genotype_phenotype.genotype_to_phenotype_mapping', _ov)
            _comp = Composite({'state': spec_pbg_pennylane_data_reuploading_composites_genotype_phenotype_genotype_to_phenotype_mapping['state']}, core=core)
            _comp.run(STEPS_quantum_4modality_6feat)
        print('ran', 'quantum_4modality_6feat', f'(pbg_pennylane_data_reuploading.composites.genotype_phenotype.genotype_to_phenotype_mapping) for', STEPS_quantum_4modality_6feat, 'steps')
    except Exception as _e:
        print(f"quantum_4modality_6feat not runnable yet: {type(_e).__name__}: {_e}")
else:
    print("RERUN=False — not re-simulating.")

## Study: Real-Data Validation — Loading and Classifying v2ecoli Zarr Stores (`v2ecoli-real-data`)

**Question.** Can the ZarrMultiOmicLoader correctly extract features from real v2ecoli zarr stores, and can the GenotypeToPhenotypeProcess classify phenotypes from this biologically realistic data?

**Objective.** Run a v2ecoli simulation with XArrayEmitter, load the zarr output with
ZarrMultiOmicLoader, map observables to modality slots, and train the
quantum classifier on real simulated multi-omic data.

**Hypothesis.** The ZarrMultiOmicLoader will correctly handle real v2ecoli zarr structure. The quantum classifier will achieve non-trivial accuracy (>0.65 AUC-ROC).


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `v2ecoli_real_data_load` | `pbg_pennylane_data_reuploading.composites.genotype_phenotype.genotype_to_phenotype_mapping` | 0 | num_omic_modalities=4, features_per_modality=3, epochs=6, batch_size=8, num_train=12, num_test=12, seed=42, noise_level=0.1, learning_rate=0.4 |


### The model & its parameters

Each composite is a process-bigraph *document* — named processes wired by `inputs`/`outputs` ports over shared stores. Below, each composite shows what it does, then a **parameter cell** with its tunable knobs (the current default and what each one means). Edit any value, then re-run the **Run** section to rebuild and re-simulate with your changes.


**The model — `pbg_pennylane_data_reuploading.composites.genotype_phenotype.genotype_to_phenotype_mapping`**

Classify multi-omic expression profiles (transcriptomics, proteomics, metabolomics, epigenomics) into viable vs non-viable cell replication using quantum data re-uploading. Each omic modality occupies one re-uploading layer.


In [ ]:
# === Parameters — edit any value, then re-run from the Run cell below ===
# Each entry is the composite's current default; the comment explains it.
ov_pbg_pennylane_data_reuploading_composites_genotype_phenotype_genotype_to_phenotype_mapping = {
    'num_omic_modalities'  : 4,   # Number of omic input modalities
    'features_per_modality': 8,   # Features per modality
    'learning_rate'        : 0.4,   # Adam learning rate
    'epochs'               : 8,   # Training epochs
    'batch_size'           : 16,   # Mini-batch size
}

### Run

_Set `STEPS`, then run. Each composite is built live from its (edited) spec above and run; a composite whose module processes aren't implemented yet reports the error instead of crashing. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: v2ecoli-real-data ===
STUDY = 'v2ecoli-real-data'
STUDY_DIR = REPO / 'workspace/studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

# This study has no *recorded* runs yet. RERUN builds its declared
# composite(s) live and runs them; a composite whose module processes
# are not implemented yet reports the error instead of crashing.
STEPS_v2ecoli_real_data_load = 100
INTERVAL_v2ecoli_real_data_load = 1.0

if RERUN:
    from process_bigraph import Composite
    try:
        with quiet():  # keep per-step stdout out of the notebook
            _ov = ov_pbg_pennylane_data_reuploading_composites_genotype_phenotype_genotype_to_phenotype_mapping if 'ov_pbg_pennylane_data_reuploading_composites_genotype_phenotype_genotype_to_phenotype_mapping' in dir() else None  # edits from the panel cell, if any
            spec_pbg_pennylane_data_reuploading_composites_genotype_phenotype_genotype_to_phenotype_mapping = materialize_composite('pbg_pennylane_data_reuploading.composites.genotype_phenotype.genotype_to_phenotype_mapping', _ov)
            _comp = Composite({'state': spec_pbg_pennylane_data_reuploading_composites_genotype_phenotype_genotype_to_phenotype_mapping['state']}, core=core)
            _comp.run(STEPS_v2ecoli_real_data_load)
        print('ran', 'v2ecoli_real_data_load', f'(pbg_pennylane_data_reuploading.composites.genotype_phenotype.genotype_to_phenotype_mapping) for', STEPS_v2ecoli_real_data_load, 'steps')
    except Exception as _e:
        print(f"v2ecoli_real_data_load not runnable yet: {type(_e).__name__}: {_e}")
else:
    print("RERUN=False — not re-simulating.")

## Open decisions
- Which classical baselines to include (RF, LR, XGBoost, SVM)?
- What is the minimum acceptable AUC-ROC for claiming quantum advantage?
- Should evaluation use cross-validation or fixed train/test split?
